# 04 - Colab GPU Training (INSEG_PRO)

This notebook is designed for **Google Colab + GPU** and trains Mask R-CNN on Cityscapes using your project code.

Execution order:
1. Runtime setup (GPU + Drive)
2. Dependencies installation
3. Dataset discovery/extraction
4. Model training (with checkpoints + resume)
5. Metrics and loss visualization

In [ ]:
import os
import sys
from pathlib import Path
import torch

print('=' * 70)
print('STEP 1 - COLAB RUNTIME CHECK')
print('=' * 70)

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory: {total_mem_gb:.2f} GB')
else:
    raise RuntimeError('No GPU detected. In Colab: Runtime > Change runtime type > GPU.')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Fixed path layout for your Drive structure.
PROJECT_DIR = Path('/content/drive/MyDrive/INSEG_PRO')
CODE_DIR = PROJECT_DIR / 'code'

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'Project folder not found at: {PROJECT_DIR}. '
        'Upload/copy INSEG_PRO to Drive and update PROJECT_DIR.'
    )

if not CODE_DIR.exists():
    raise FileNotFoundError(
        f'Code folder not found at: {CODE_DIR}. '
        'Expected INSEG_PRO/code in Google Drive.'
    )

# Your dataset loader is at INSEG_PRO/code/dataset_loader.py.
DATASET_LOADER_PATH = CODE_DIR / 'dataset_loader.py'
if not DATASET_LOADER_PATH.exists():
    raise FileNotFoundError(
        f'dataset_loader.py not found at: {DATASET_LOADER_PATH}'
    )

%cd {CODE_DIR}
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print('Project directory:', PROJECT_DIR)
print('Code directory   :', CODE_DIR)
print('dataset_loader   :', DATASET_LOADER_PATH)
print('requirements in code dir   :', (CODE_DIR / 'requirements.txt').exists())
print('requirements in project dir:', (PROJECT_DIR / 'requirements.txt').exists())

In [ ]:
print('=' * 70)
print('STEP 2 - INSTALL DEPENDENCIES')
print('=' * 70)

import sys
import site
import pkgutil
import subprocess
import importlib.machinery
from pathlib import Path

# Compatibility shims for old pkg_resources implementations on newer Python.
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = pkgutil.zipimporter
if not hasattr(importlib.machinery.FileFinder, 'find_module'):
    def _ff_find_module(self, fullname):
        spec = self.find_spec(fullname)
        return spec.loader if spec else None
    importlib.machinery.FileFinder.find_module = _ff_find_module

# Prefer /usr/local site-packages over distro packages.
for p in reversed([x for x in site.getsitepackages() if x.startswith('/usr/local')]):
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

REQ_FILE_CODE = CODE_DIR / 'requirements.txt'
REQ_FILE_PROJECT = PROJECT_DIR / 'requirements.txt'
REQ_FILE = REQ_FILE_CODE if REQ_FILE_CODE.exists() else REQ_FILE_PROJECT

def run_pip(args):
    cmd = [sys.executable, '-m', 'pip'] + args
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# Bootstrap packaging tools.
run_pip(['install', '-q', '--upgrade', 'pip', 'wheel', 'setuptools>=75.0.0', 'packaging', 'jedi'])

installed_from_requirements = False
if REQ_FILE.exists():
    print(f'Using requirements file: {REQ_FILE}')
    try:
        run_pip(['install', '-q', '-r', str(REQ_FILE)])
        installed_from_requirements = True
    except subprocess.CalledProcessError:
        print('requirements.txt install failed, switching to fallback dependency set...')
else:
    print('No requirements.txt found in code/ or project root.')

if not installed_from_requirements:
    fallback_packages = [
        'torch>=2.0.0',
        'torchvision>=0.15.0',
        'torchaudio>=2.0.0',
        'opencv-python>=4.8.0',
        'Pillow>=10.0.0',
        'albumentations>=1.3.0',
        'numpy>=1.24.0',
        'pandas>=2.0.0',
        'matplotlib>=3.7.0',
        'seaborn>=0.12.0',
        'tqdm>=4.65.0',
        'pyyaml>=6.0',
        'pycocotools>=2.0.0',
        'ipywidgets>=8.0.0'
    ]
    run_pip(['install', '-q', *fallback_packages])

# Install Detectron2.
run_pip(['install', '-q', '--no-build-isolation', 'git+https://github.com/facebookresearch/detectron2.git'])

# Re-apply shims before import checks.
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = pkgutil.zipimporter
if not hasattr(importlib.machinery.FileFinder, 'find_module'):
    def _ff_find_module(self, fullname):
        spec = self.find_spec(fullname)
        return spec.loader if spec else None
    importlib.machinery.FileFinder.find_module = _ff_find_module

import detectron2
from detectron2 import model_zoo
print('Detectron2 installed:', detectron2.__version__)
print('Detectron2 import test: OK')

In [ ]:
print('=' * 70)
print('STEP 3 - DATASET PREP')
print('=' * 70)

import zipfile

DATA_DIR = CODE_DIR / 'data' / 'raw'
LEFT_DIR = DATA_DIR / 'leftImg8bit'
GT_DIR = DATA_DIR / 'gtFine'

ZIP_CANDIDATES = [
    PROJECT_DIR / 'data_zips',
    PROJECT_DIR / 'code' / 'data_zips',
    CODE_DIR / 'data_zips',
    CODE_DIR / 'data' / 'download_scripts',
    Path('/content/drive/MyDrive/data_zips')
]

if LEFT_DIR.exists() and GT_DIR.exists():
    print('Dataset already extracted.')
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_dir = None
    for candidate in ZIP_CANDIDATES:
        if candidate.exists():
            if (candidate / 'leftImg8bit_trainvaltest.zip').exists() and (candidate / 'gtFine_trainvaltest.zip').exists():
                zip_dir = candidate
                break

    if zip_dir is None:
        raise FileNotFoundError(
            'Could not find Cityscapes zip files. Expected leftImg8bit_trainvaltest.zip and '
            'gtFine_trainvaltest.zip in one of: ' + ', '.join(str(p) for p in ZIP_CANDIDATES)
        )

    print('Using zip directory:', zip_dir)
    for zip_name in ['leftImg8bit_trainvaltest.zip', 'gtFine_trainvaltest.zip']:
        zip_path = zip_dir / zip_name
        print(f'Extracting {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA_DIR)

    print('Extraction complete.')

print('Data root:', DATA_DIR)
print('leftImg8bit exists:', LEFT_DIR.exists())
print('gtFine exists:', GT_DIR.exists())

In [ ]:
print('=' * 70)
print('STEP 4 - LOAD DATASET')
print('=' * 70)

import sys
import json
import shutil
import pkgutil
import importlib
import importlib.machinery
import pandas as pd
import matplotlib.pyplot as plt

# Safety patches for old pkg_resources in some Colab runtimes.
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = pkgutil.zipimporter
if not hasattr(importlib.machinery.FileFinder, 'find_module'):
    def _ff_find_module(self, fullname):
        spec = self.find_spec(fullname)
        return spec.loader if spec else None
    importlib.machinery.FileFinder.find_module = _ff_find_module

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print('Using code root:', CODE_DIR)
print('dataset_loader path:', CODE_DIR / 'dataset_loader.py')

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, HookBase
from detectron2.evaluation import COCOEvaluator
from detectron2.utils.events import get_event_storage

# Your structure: INSEG_PRO/code/dataset_loader.py
dataset_loader = importlib.import_module('dataset_loader')
register_cityscapes_dataset = dataset_loader.register_cityscapes_dataset
CITYSCAPES_THING_CLASSES = dataset_loader.CITYSCAPES_THING_CLASSES

train_dataset_name, train_dicts, _ = register_cityscapes_dataset(DATA_DIR, split='train')
val_dataset_name, val_dicts, _ = register_cityscapes_dataset(DATA_DIR, split='val')

print('Train images:', len(train_dicts))
print('Val images:', len(val_dicts))
print('Num classes:', len(CITYSCAPES_THING_CLASSES))

In [ ]:
print('=' * 70)
print('STEP 5 - CONFIGURE MODEL')
print('=' * 70)

OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'mask_rcnn_cityscapes_colab'
CHECKPOINT_DIR = PROJECT_DIR / 'outputs' / 'checkpoints_colab'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file('COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml')
)
cfg.DATASETS.TRAIN = (train_dataset_name,)
cfg.DATASETS.TEST = (val_dataset_name,)
cfg.DATALOADER.NUM_WORKERS = 2

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
)
cfg.MODEL.DEVICE = 'cuda'
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CITYSCAPES_THING_CLASSES)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256

cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 1e-4
cfg.SOLVER.MAX_ITER = 10000
cfg.SOLVER.STEPS = (6000, 8500)
cfg.SOLVER.GAMMA = 0.1
cfg.SOLVER.WARMUP_ITERS = 500
cfg.SOLVER.WARMUP_FACTOR = 0.001
cfg.SOLVER.CHECKPOINT_PERIOD = 300

cfg.TEST.EVAL_PERIOD = 500
cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)
cfg.INPUT.MAX_SIZE_TRAIN = 1333
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.INPUT.RANDOM_FLIP = 'horizontal'

cfg.OUTPUT_DIR = str(OUTPUT_DIR)

print('Output dir:', OUTPUT_DIR)
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Iterations:', cfg.SOLVER.MAX_ITER)
print('Checkpoint period:', cfg.SOLVER.CHECKPOINT_PERIOD)
print('STEP 5 done')

In [ ]:
print('=' * 70)
print('STEP 6 - TRAINER WITH BACKUP HOOKS')
print('=' * 70)

class CheckpointBackupHook(HookBase):
    def __init__(self, checkpoint_dir: Path, backup_period: int = 300):
        super().__init__()
        self.checkpoint_dir = Path(checkpoint_dir)
        self.backup_period = backup_period
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def after_step(self):
        if (self.trainer.iter + 1) % self.backup_period != 0:
            return

        iteration = self.trainer.iter + 1
        ckpt_src = Path(self.trainer.cfg.OUTPUT_DIR) / f'model_{iteration:07d}.pth'
        if ckpt_src.exists():
            ckpt_dst = self.checkpoint_dir / ckpt_src.name
            shutil.copy2(ckpt_src, ckpt_dst)
            print(f'[Backup] {ckpt_dst.name}')


class LossTrackerHook(HookBase):
    def __init__(self):
        super().__init__()
        self.losses = []

    def after_step(self):
        storage = get_event_storage()
        latest = storage.latest()
        loss_dict = {}
        for k, v in latest.items():
            if 'loss' in k:
                loss_dict[k] = v[0] if isinstance(v, tuple) else v

        if loss_dict:
            self.losses.append(loss_dict)
            if self.trainer.iter % 50 == 0:
                total_loss = sum(loss_dict.values())
                print(f'[Iter {self.trainer.iter:5d}] total_loss={total_loss:.4f}')


class ImprovedTrainer(DefaultTrainer):
    def __init__(self, cfg):
        self.loss_hook = LossTrackerHook()
        self.backup_hook = CheckpointBackupHook(
            CHECKPOINT_DIR,
            backup_period=cfg.SOLVER.CHECKPOINT_PERIOD
        )
        super().__init__(cfg)

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'inference')
        return COCOEvaluator(dataset_name, cfg, False, output_folder)

    def build_hooks(self):
        hooks = super().build_hooks()
        hooks.insert(-1, self.loss_hook)
        hooks.insert(-1, self.backup_hook)
        return hooks

print('Trainer class ready.')

In [ ]:
print('=' * 70)
print('STEP 7 - START OR RESUME TRAINING')
print('=' * 70)

trainer = ImprovedTrainer(cfg)

# Auto-resume from the latest available checkpoint to avoid losing progress.
AUTO_RESUME = True

if AUTO_RESUME:
    candidates = list(OUTPUT_DIR.glob('model_*.pth')) + list(CHECKPOINT_DIR.glob('model_*.pth'))
    if candidates:
        def _iter_from_name(path_obj):
            try:
                return int(path_obj.stem.split('_')[-1])
            except Exception:
                return -1

        latest = max(candidates, key=_iter_from_name)
        target = OUTPUT_DIR / latest.name
        if latest != target:
            shutil.copy2(latest, target)

        # Ensure Detectron2 knows the latest checkpoint to resume from.
        with open(OUTPUT_DIR / 'last_checkpoint', 'w') as f:
            f.write(latest.name)

        print(f'Auto-resume enabled. Latest checkpoint: {latest.name}')
        trainer.resume_or_load(resume=True)
    else:
        print('No checkpoint found. Starting from COCO weights.')
        trainer.resume_or_load(resume=False)
else:
    trainer.resume_or_load(resume=False)

trainer.train()

final_model = OUTPUT_DIR / 'model_final.pth'
if final_model.exists():
    shutil.copy2(final_model, CHECKPOINT_DIR / final_model.name)
    print('Final model backed up to checkpoints directory.')

In [ ]:
print('=' * 70)
print('STEP 8 - METRICS AND LOSS PLOTS')
print('=' * 70)

metrics_file = OUTPUT_DIR / 'metrics.json'
if metrics_file.exists():
    rows = []
    with open(metrics_file, 'r') as f:
        for line in f:
            rows.append(json.loads(line))

    val_rows = [r for r in rows if 'bbox/AP' in r]
    if val_rows:
        best = max(val_rows, key=lambda x: x.get('bbox/AP', 0.0))
        print('Best validation checkpoint:')
        print('  iteration:', best.get('iteration'))
        print('  bbox/AP :', round(best.get('bbox/AP', 0.0), 3))
        print('  segm/AP :', round(best.get('segm/AP', 0.0), 3))
    else:
        print('No validation rows with bbox/AP found yet.')
else:
    print('metrics.json not found. Run training first.')

if 'trainer' in locals() and hasattr(trainer, 'loss_hook') and trainer.loss_hook.losses:
    loss_df = pd.DataFrame(trainer.loss_hook.losses)
    loss_cols = [c for c in loss_df.columns if 'loss' in c]
    if loss_cols:
        loss_df['total_loss'] = loss_df[loss_cols].sum(axis=1)

    plt.figure(figsize=(12, 5))
    for col in ['loss_cls', 'loss_box_reg', 'loss_mask', 'loss_rpn_cls', 'loss_rpn_loc', 'total_loss']:
        if col in loss_df.columns:
            plt.plot(loss_df.index, loss_df[col], label=col, linewidth=1.5)

    plt.title('Training Loss Curves')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plot_path = OUTPUT_DIR / 'training_losses_colab.png'
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print('Saved loss plot:', plot_path)
else:
    print('Loss history unavailable in current session.')

## Notes

- Checkpoints are saved and backed up every 300 iterations.
- Auto-resume is enabled by default in Step 7 and loads the latest checkpoint automatically.
- Keep project/output folders in Google Drive so checkpoints persist across sessions.
- If Detectron2 install fails after runtime changes, restart runtime and run from Step 2 again.